In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, to_timestamp, explode, row_number
from pyspark.sql.window import Window

# 1. Setup Variables
catalog = "maritime_ais"
bronze_schema = "maritime_bronze"
silver_schema = "maritime_silver"
checkpoint_base = "abfss://maritime-lake@maritimepipeline.dfs.core.windows.net/checkpoints/silver"

# 2. Define Upsert Logic (Spark Connect safe + dedup)
def upsert_to_silver(microBatchDF, batchId, table_name, merge_condition, dedup_keys=None):
    try:
        _spark = microBatchDF.sparkSession

        # Deduplicate source rows so at most one row per merge key survives
        if dedup_keys:
            w = Window.partitionBy(*dedup_keys).orderBy(col(dedup_keys[0]))
            microBatchDF = (
                microBatchDF.withColumn("_rn", row_number().over(w))
                .filter(col("_rn") == 1)
                .drop("_rn")
            )

        if not _spark.catalog.tableExists(table_name):
            microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
            return

        delta_table = DeltaTable.forName(_spark, table_name)
        delta_table.alias("target").merge(
            microBatchDF.alias("source"),
            merge_condition
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

    except Exception as e:
        print(f"CRITICAL ERROR in batch {batchId}: {str(e)}")
        raise e

# 3. Process AIS Locations
print("Processing Silver AIS Locations...")
df_bronze_ais = spark.readStream.table(f"{catalog}.{bronze_schema}.ais_locations")

df_exploded_ais = df_bronze_ais.select(explode(col("features")).alias("feature"))

df_silver_ais = df_exploded_ais.select(
    col("feature.properties.mmsi").cast("string").alias("mmsi"),
    col("feature.geometry.coordinates").getItem(0).cast("double").alias("longitude"),
    col("feature.geometry.coordinates").getItem(1).cast("double").alias("latitude"),
    col("feature.properties.sog").cast("double").alias("speed_over_ground"),
    col("feature.properties.cog").cast("double").alias("course_over_ground"),
    col("feature.properties.heading").cast("double").alias("heading"),
    (col("feature.properties.timestampExternal") / 1000).cast("timestamp").alias("timestamp")
)

ais_table = f"{catalog}.{silver_schema}.ais_locations"

# Composite key preserves history while handling exact duplicates
composite_merge_condition = "target.mmsi = source.mmsi AND target.timestamp = source.timestamp"

query_ais = (
    df_silver_ais.writeStream
    .foreachBatch(lambda df, epoch_id: upsert_to_silver(
        df, epoch_id, ais_table, composite_merge_condition, dedup_keys=["mmsi", "timestamp"]
    ))
    .option("checkpointLocation", f"{checkpoint_base}/ais_locations")
    .trigger(availableNow=True)
    .start()
)
query_ais.processAllAvailable()
print("Completed AIS Processing.")